In [ ]:
from pathlib import Path
import sys

sys.path.insert(0, str(Path.cwd() / 'src'))

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from datetime import datetime

from risk_engine.data import load_adjusted_prices
from risk_engine.returns import calculate_portfolio_returns, calculate_simple_returns
from risk_engine.var import (
    historical_cvar,
    historical_var,
    monte_carlo_normal_var,
    parametric_cvar,
    parametric_var,
)
from risk_engine.stress import run_stress_scenario
from risk_engine.factors import run_pca
from risk_engine.backtest import backtest_var, rolling_historical_var

print('Risk engine modules loaded successfully')

In [ ]:
# Configuration
tickers = ['SPY', 'TLT', 'GLD', 'XLE', 'EEM']
weights = np.array([0.40, 0.25, 0.15, 0.10, 0.10])
start = '2008-01-01'
end = datetime.today().strftime('%Y-%m-%d')
confidence = 0.95
alpha = 1 - confidence
n_simulations = 100000
random_seed = 42
window = 252

scenarios = {
    '2008 Financial Crisis': ('2008-09-01', '2009-03-31'),
    'COVID Crash': ('2020-02-19', '2020-03-23'),
    '2022 Rate Hikes': ('2022-01-01', '2022-12-31'),
}

print(f'Downloading data from {start} to {end}...')
prices = load_adjusted_prices(tickers, start, end)
returns = calculate_simple_returns(prices)
portfolio_returns = calculate_portfolio_returns(returns, weights)

print(f'\nShape: {prices.shape}')
print(f'Date range: {prices.index[0].date()} to {prices.index[-1].date()}')
print(f'\nWeights: {dict(zip(tickers, weights))}')
print(f'Weight sum: {weights.sum()}')
print('\nPortfolio daily return stats:')
print(f'  Mean:  {portfolio_returns.mean() * 100:.3f}%')
print(f'  Stdev: {portfolio_returns.std() * 100:.3f}%')

In [ ]:
# Value at Risk: historical, parametric, and Monte Carlo
hist_var = historical_var(portfolio_returns, confidence)
hist_cvar = historical_cvar(portfolio_returns, confidence)
param_var = parametric_var(portfolio_returns, confidence)
param_cvar = parametric_cvar(portfolio_returns, confidence)
mc_var = monte_carlo_normal_var(
    portfolio_returns,
    confidence=confidence,
    n_simulations=n_simulations,
    seed=random_seed,
)

# The existing module exposes Monte Carlo VaR. Recreate the same seeded sample
# here only to display the notebook's existing Monte Carlo CVaR and histogram.
mc_rng = np.random.RandomState(random_seed)
mc_returns = pd.Series(
    mc_rng.normal(portfolio_returns.mean(), portfolio_returns.std(), n_simulations)
)
mc_cvar = mc_returns[mc_returns <= mc_var].mean()

print('=' * 55)
print('   VALUE AT RISK — THREE METHODS (95% Confidence)')
print('=' * 55)
print(f'{"Method":<25} {"VaR":>10} {"CVaR":>10}')
print('-' * 55)
print(f'{"Historical Simulation":<25} {hist_var * 100:>9.3f}% {hist_cvar * 100:>9.3f}%')
print(f'{"Parametric (Normal)":<25} {param_var * 100:>9.3f}% {param_cvar * 100:>9.3f}%')
print(f'{"Monte Carlo":<25} {mc_var * 100:>9.3f}% {mc_cvar * 100:>9.3f}%')
print('=' * 55)
print('\nInterpretation: On any given day, there is a 5% chance')
print(f'the portfolio loses more than ~{abs(hist_var) * 100:.2f}% (historical VaR)')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

methods = [
    ('Historical Simulation', portfolio_returns, hist_var, hist_cvar, '#2196F3'),
    ('Parametric (Normal)', portfolio_returns, param_var, param_cvar, '#4CAF50'),
    ('Monte Carlo', mc_returns, mc_var, mc_cvar, '#FF9800'),
]

for ax, (title, data, var, cvar, color) in zip(axes, methods):
    ax.hist(data, bins=100, alpha=0.6, color=color, density=True)
    ax.axvline(var, color='red', linestyle='--', linewidth=1.5, label=f'VaR: {var * 100:.2f}%')
    ax.axvline(cvar, color='darkred', linestyle='--', linewidth=1.5, label=f'CVaR: {cvar * 100:.2f}%')
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_xlabel('Daily Return')
    ax.set_ylabel('Density')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle('VaR & CVaR — Three Methods Comparison', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('var_comparison.png', dpi=150, bbox_inches='tight')
print('Saved: var_comparison.png')

In [ ]:
# Historical stress scenarios
print('=' * 65)
print('   STRESS TEST — PORTFOLIO PERFORMANCE IN CRISIS PERIODS')
print('=' * 65)
print(f'{"Scenario":<25} {"Period":<25} {"Return":>8} {"Max DD":>8} {"VaR(95)":>8}')
print('-' * 65)

scenario_results = {}
for name, (start_date, end_date) in scenarios.items():
    result = run_stress_scenario(portfolio_returns, start_date, end_date, confidence)
    scenario_results[name] = result
    period_str = f'{start_date} to {end_date[:7]}'
    print(f'{name:<25} {period_str:<25} {result["cum_ret"] * 100:>7.1f}% {result["max_dd"] * 100:>7.1f}% {result["var"] * 100:>7.2f}%')

print('=' * 65)
print(f'\nBaseline (full period) VaR: {hist_var * 100:.2f}%')
print('Note: Crisis VaR >> baseline VaR — models trained on calm')
print('periods systematically underestimate tail risk in crises.')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
colors = ['#F44336', '#FF9800', '#9C27B0']

for ax, (name, data), color in zip(axes, scenario_results.items(), colors):
    crisis_returns = data['returns']
    cum_curve = (1 + crisis_returns).cumprod()
    ax.plot(cum_curve.index, cum_curve.values, color=color, linewidth=1.5)
    ax.fill_between(cum_curve.index, cum_curve.values, 1, where=(cum_curve.values < 1), alpha=0.3, color=color)
    ax.axhline(y=1, color='gray', linestyle=':', linewidth=0.8)
    ax.set_title(f'{name}\nReturn: {data["cum_ret"] * 100:.1f}% | Max DD: {data["max_dd"] * 100:.1f}%', fontsize=10, fontweight='bold')
    ax.set_xlabel('Date')
    ax.set_ylabel('Portfolio Value ($1 invested)')
    ax.grid(True, alpha=0.3)
    ax.tick_params(axis='x', rotation=30)

plt.suptitle('Portfolio Stress Test — Crisis Period Performance', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('stress_test.png', dpi=150, bbox_inches='tight')
print('Saved: stress_test.png')

In [ ]:
# PCA factor decomposition
pca_results = run_pca(returns)
explained_var = pca_results['explained_variance_ratio']
cumulative_var = explained_var.cumsum()
loadings = pca_results['loadings']

print('=' * 50)
print('   PCA FACTOR DECOMPOSITION')
print('=' * 50)
print(f'{"Factor":<10} {"Explained Var":>15} {"Cumulative":>12}')
print('-' * 50)
for factor, (ev, cv) in enumerate(zip(explained_var, cumulative_var), start=1):
    print(f'{"PC" + str(factor):<10} {ev * 100:>14.1f}% {cv * 100:>11.1f}%')
print('=' * 50)
print('\nFactor Loadings (how each asset contributes to each factor):')
print(loadings.round(3).to_string())
print(f'\nPC1 explains {explained_var.iloc[0] * 100:.1f}% of portfolio variance')
print(f'PC1+PC2 explain {cumulative_var.iloc[1] * 100:.1f}% of portfolio variance')
print('\nInterpretation: PC1 is likely the broad market factor —')
print('high loadings on SPY, EEM, XLE and negative on TLT (flight to safety)')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.bar(range(1, len(explained_var) + 1), explained_var * 100, color='#2196F3', alpha=0.7, label='Individual')
ax1.plot(range(1, len(cumulative_var) + 1), cumulative_var * 100, 'ro-', linewidth=2, label='Cumulative')
ax1.axhline(y=90, color='gray', linestyle='--', linewidth=0.8, label='90% threshold')
ax1.set_xlabel('Principal Component')
ax1.set_ylabel('Variance Explained (%)')
ax1.set_title('Scree Plot — Variance Explained by Each Factor', fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

sns.heatmap(loadings[['PC1', 'PC2', 'PC3']], annot=True, fmt='.3f', cmap='RdYlGn', center=0, vmin=-1, vmax=1, ax=ax2)
ax2.set_title('Factor Loadings — First 3 Principal Components', fontweight='bold')
ax2.set_xlabel('Factor')
ax2.set_ylabel('Asset')

plt.tight_layout()
plt.savefig('pca_decomposition.png', dpi=150, bbox_inches='tight')
print('Saved: pca_decomposition.png')

In [ ]:
# VaR backtesting
rolling_var = rolling_historical_var(portfolio_returns, window=window, confidence=confidence)
actual_returns = portfolio_returns.loc[rolling_var.index]
backtest_results = backtest_var(actual_returns, rolling_var)
exceptions = backtest_results['exceptions']
exception_rate = backtest_results['exception_rate']
classification = backtest_results['classification']

print('=' * 55)
print('   VAR BACKTEST RESULTS (Rolling 252-day Window)')
print('=' * 55)
print(f'  Test period:        {rolling_var.index[0].date()} to {rolling_var.index[-1].date()}')
print(f'  Total trading days: {backtest_results["observation_count"]}')
print(f'  VaR exceptions:     {backtest_results["exception_count"]} days')
print(f'  Exception rate:     {exception_rate * 100:.2f}% (target: 5.00%)')
print('  Model status:       ', end='')
status_messages = {
    'acceptable': '✓ ACCEPTABLE — exceptions within Basel threshold',
    'yellow': '⚠ YELLOW ZONE — elevated exceptions, under review',
    'red': '✗ RED ZONE — model rejected, recalibration required',
}
print(status_messages[classification])
print('=' * 55)
print('\nBasel III Traffic Light:')
print('  Green zone:  0–4 exceptions per year → model accepted')
print('  Yellow zone: 5–9 exceptions → increased scrutiny')
print('  Red zone:    10+ exceptions → model rejected')

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), gridspec_kw={'height_ratios': [2, 1]})

ax1.plot(actual_returns.index, actual_returns.values * 100, color='#2196F3', linewidth=0.5, alpha=0.7, label='Actual daily return')
ax1.plot(rolling_var.index, rolling_var.values * 100, color='#F44336', linewidth=1.2, label='Rolling VaR (95%)')
ax1.scatter(actual_returns[exceptions].index, actual_returns[exceptions].values * 100, color='red', s=8, zorder=5, label=f'Exceptions ({backtest_results["exception_count"]} days)')
ax1.axhline(y=0, color='gray', linestyle=':', linewidth=0.8)
ax1.set_ylabel('Daily Return (%)')
ax1.set_title('VaR Backtest — Actual Returns vs Rolling VaR (2009–2026)', fontsize=12, fontweight='bold')
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)

rolling_exception_rate = exceptions.rolling(window).mean() * 100
ax2.plot(rolling_exception_rate.index, rolling_exception_rate.values, color='#FF9800', linewidth=1.2, label='Rolling exception rate (252-day)')
ax2.axhline(y=5, color='red', linestyle='--', linewidth=1, label='5% Basel threshold')
ax2.axhline(y=8, color='darkred', linestyle='--', linewidth=1, label='8% red zone')
ax2.fill_between(rolling_exception_rate.index, rolling_exception_rate.values, 5, where=(rolling_exception_rate.values > 5), alpha=0.3, color='red', label='Above threshold')
ax2.set_ylabel('Exception Rate (%)')
ax2.set_xlabel('Date')
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('var_backtest.png', dpi=150, bbox_inches='tight')
print('Saved: var_backtest.png')